# 📘 Supervised Learning – Regression (Improved Version)
### Walmart Monthly Sales Forecasting with Ridge Regression and Log-Transformation
> Improves accuracy for large sales values and avoids underfitting by transforming sales and using monthly trends.

In [ ]:
# --- Setup and Imports ---
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import Ridge
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

In [ ]:
# --- Load Merged Walmart Dataset ---
df = pd.read_csv('walmart_sales.csv')

# Drop extra index column if present
if 'Unnamed: 0' in df.columns:
    df.drop(columns='Unnamed: 0', inplace=True)

# Convert Date column to datetime for time-based grouping later
df['Date'] = pd.to_datetime(df['Date'])
df.head()

In [ ]:
# --- Select Features and Target ---
features = [
    'Store', 'Dept', 'Fuel_Price', 'IsHoliday', 'CPI', 'Unemployment',
    'Temperature', 'Size', 'month', 'Super_Bowl', 'Labor_Day', 'Thanksgiving', 'Christmas'
]
X = df[features].fillna(0)
X['IsHoliday'] = X['IsHoliday'].astype(int)
y = df['Weekly_Sales']

In [ ]:
# --- Apply Log Transformation to Target ---
y_log = np.log1p(y)

In [ ]:
# --- Train/Test Split ---
X_train, X_test, y_train_log, y_test_log = train_test_split(X, y_log, test_size=0.2, random_state=42)

In [ ]:
# --- Train Ridge Regression Model ---
model = Ridge(alpha=1.0)
model.fit(X_train, y_train_log)

In [ ]:
# --- Predict and Convert Back to Original Scale ---
y_pred_log = model.predict(X_test)
y_pred = np.expm1(y_pred_log)
y_test = np.expm1(y_test_log)

In [ ]:
# --- Evaluate Model ---
mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(np.mean((y_test - y_pred) ** 2))
r2 = r2_score(y_test, y_pred)

print(f"Mean Absolute Error (MAE): {mae:.2f}")
print(f"Root Mean Squared Error (RMSE): {rmse:.2f}")
print(f"R² Score: {r2:.2f}")

In [ ]:
# --- Visualize Monthly Aggregated Actual vs Predicted Sales ---
date_series = df.loc[X_test.index, 'Date']
result_df = pd.DataFrame({
    'Date': pd.to_datetime(date_series),
    'Actual_Sales': y_test,
    'Predicted_Sales': y_pred
}).sort_values('Date')

# Resample to monthly by summing sales
monthly_df = result_df.set_index('Date').resample('M').sum().reset_index()

plt.figure(figsize=(12, 6))
plt.plot(monthly_df['Date'], monthly_df['Actual_Sales'], label='Actual Sales', marker='o')
plt.plot(monthly_df['Date'], monthly_df['Predicted_Sales'], label='Predicted Sales', marker='x')
plt.xlabel("Month")
plt.ylabel("Total Monthly Sales")
plt.title("Walmart Monthly Sales Forecast: Actual vs Predicted")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

### 💡 Optional Enhancements
- Use Store/Dept-specific models for better performance
- Add rolling averages or seasonality features
- Try models like XGBoost or Random Forest for non-linearity